In [44]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.express as px
import category_encoders as ce
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None) #to show all columns when we print the dataframe

In [21]:
df = pd.read_csv('../LFB_project_data/distance_calc/dataset_with_filtered_distance_speed.csv')

df.head()


,index,IncidentNumber,CalYear,Month,Weekday,Hour,Is_Nightshift,Is_Rush_Hour,Is_Weekend,Is_Public_Holiday,AttendanceTimeSeconds,DeployedFromStation_Name,PumpOrder,IncidentGroup,Is_SpecialService,SpecialServiceType,PropertyCategory,PropertyType,IncGeo_BoroughName,NumCalls,Is_RepeatedCall,Latitude,Longitude,distance_fire_to_station,avg_speed,NumOfCalls_bucket
0,0,000004-01012021,2021,1,4,0,1,0,0,1,181,HORNSEY,1,False Alarm,0,NoSpecialService,Non Residential,Single shop,HARINGEY,1.0,0,51.590249,-0.143203,1490.3,40.338947,1
1,1,000005-01012021,2021,1,4,0,1,0,0,1,250,WOODFORD,1,Fire,0,NoSpecialService,Outdoor,Tree scrub,REDBRIDGE,1.0,0,51.606914,0.030447,1166.4,30.875294,1
2,2,000006-01012021,2021,1,4,0,1,0,0,1,376,DAGENHAM,1,False Alarm,0,NoSpecialService,Outdoor,Road surface/pavement,BARKING AND DAGENHAM,1.0,0,51.540143,0.154479,3463.6,50.893714,1
3,3,000007-01012021,2021,1,4,0,1,0,0,1,409,WANDSWORTH,1,False Alarm,0,NoSpecialService,Dwelling,Purpose Built Flats/Maisonettes - 4 to 9 storeys,WANDSWORTH,4.0,1,51.446439,-0.214895,1975.6,19.977978,4-5
4,4,000009-01012021,2021,1,4,0,1,0,0,1,362,STRATFORD,1,Fire,0,NoSpecialService,Road Vehicle,Car,NEWHAM,2.0,1,51.547241,0.023549,1049.4,14.757188,2


In [30]:
# number of calls, outlier management - buckets 
def bucket_num_calls(x):
            if x <= 1:
                return "1"
            elif x == 2:
                return "2"
            elif x == 3:
                return "3"
            elif x <= 5:
                return "4-5"
            elif x <= 10:
                return "6-10"
            else:
                return "10+"
col = "NumCalls"  
df[col] = pd.to_numeric(df[col], errors='coerce').fillna(1)
df["NumOfCalls_bucket"] = df[col].apply(bucket_num_calls)


In [35]:
#Delete Column NumCalls

df = df.drop(columns=["NumCalls"])



In [ ]:
# Is_central_London as new column
inner_london = [
    'CITY OF LONDON', 'WESTMINSTER', 'CAMDEN', 'ISLINGTON', 'HACKNEY', 
    'TOWER HAMLETS', 'SOUTHWARK', 'LAMBETH', 'KENSINGTON AND CHELSEA', 
    'HAMMERSMITH AND FULHAM', 'WANDSWORTH', 'LEWISHAM', 'NEWHAM', 'HARINGEY'
]

# 2. make binary flag (1 for Inner, 0 for Outer)
df['Is_central_London'] = df['IncGeo_BoroughName'].isin(inner_london).astype(int)


In [45]:
#Splitting the data into train and test

# --- KONFIGURATION for columns ---
TARGET = "AttendanceTimeSeconds"

SPLIT_YEAR = 2025 

# drop this colums as features

DROP_COLS = ["index", "IncidentNumber", "avg_speed"] 

# --- SPLIT ---

# Training: everything before 2025 | Test: everything after 2025
train_df = df[df["CalYear"] < SPLIT_YEAR].copy()
test_df  = df[df["CalYear"] >= SPLIT_YEAR].copy()

def get_baseline_data(data):
    # 1. drop target and drop_cols for X
    to_drop = [TARGET] + DROP_COLS
    X = data.drop(columns=[c for c in to_drop if c in data.columns])
    
    y = data[TARGET]
    y_log = np.log1p(y)
    
    return X, y, y_log

X_train, y_train, y_train_log = get_baseline_data(train_df)
X_test, y_test, y_test_log = get_baseline_data(test_df)

print(f"Done! Features in Modell: {list(X_train.columns)}")
print(f"Train Size: {len(X_train)}, Test Size: {len(X_test)}")

Done! Features in Modell: ['CalYear', 'Month', 'Weekday', 'Hour', 'Is_Nightshift', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Public_Holiday', 'DeployedFromStation_Name', 'PumpOrder', 'IncidentGroup', 'Is_SpecialService', 'SpecialServiceType', 'PropertyCategory', 'PropertyType', 'IncGeo_BoroughName', 'Is_RepeatedCall', 'Latitude', 'Longitude', 'distance_fire_to_station', 'NumOfCalls_bucket', 'Is_central_London']
Train Size: 426881, Test Size: 137825


In [46]:
# encoding

feature_config = {
    # Zeitliche Daten (Cyclic)
    "Month": {"encoding": "CYCLIC"},
    "Weekday": {"encoding": "CYCLIC"},
    "Hour": {"encoding": "CYCLIC"},
    
    # Binäre & Numerische Daten (Keep)
    "CalYear": {"encoding": "NUMERIC_KEEP"},
    "Is_Nightshift": {"encoding": "BINARY_KEEP"},
    "Is_Rush_Hour": {"encoding": "BINARY_KEEP"},
    "Is_Weekend": {"encoding": "BINARY_KEEP"},
    "Is_Public_Holiday": {"encoding": "BINARY_KEEP"},
    "Is_SpecialService": {"encoding": "BINARY_KEEP"},
    "Is_RepeatedCall": {"encoding": "BINARY_KEEP"},
    "Is_central_London": {"encoding": "BINARY_KEEP"},
    "Latitude": {"encoding": "NUMERIC_KEEP"},
    "Longitude": {"encoding": "NUMERIC_KEEP"},
    "distance_fire_to_station": {"encoding": "NUMERIC_KEEP"},
    "Is_central_London": {"encoding": "BINARY_KEEP"},
    
    # Kategorische Daten (One-Hot / Top-N)
    "IncidentGroup": {"encoding": "ONE_HOT"},
    "PropertyCategory": {"encoding": "ONE_HOT"},
    "NumOfCalls_bucket": {"encoding": "ONE_HOT"},
    "SpecialServiceType": {"encoding": "TOP_N_PLUS_ONE_HOT", "top_n": 10},
    
    # Target Encoding (Leave-One-Out)
    "DeployedFromStation_Name": {"encoding": "LEAVE_ONE_OUT_TARGET"},
    "PropertyType": {"encoding": "LEAVE_ONE_OUT_TARGET"},
    "IncGeo_BoroughName": {"encoding": "LEAVE_ONE_OUT_TARGET"}
}

# columns to drop
DROP_COLS = ["PumpOrder"]


def apply_encoding(X, y, config, is_train=True, fitted_encoders=None):
    if is_train:
        fitted_encoders = {}
    
    encoded_parts = []
    
    for col, cfg in config.items(): # cfg ist jetzt das Dictionary, z.B. {"encoding": "CYCLIC"}
        if col not in X.columns: continue
        
        method = cfg["encoding"] # Hier holen wir uns den String ("CYCLIC", "ONE_HOT" etc.)
        
        if method in ["NUMERIC_KEEP", "BINARY_KEEP"]:
            encoded_parts.append(X[[col]])
            
        elif method == "CYCLIC":
            periods = {"Month": 12, "Weekday": 7, "Hour": 24}
            p = periods[col]
            encoded_parts.append(pd.DataFrame({
                f"{col}_sin": np.sin(2 * np.pi * X[col] / p),
                f"{col}_cos": np.cos(2 * np.pi * X[col] / p)
            }, index=X.index))
            
        elif method == "ONE_HOT" or method == "TOP_N_PLUS_ONE_HOT":
            if is_train:
                # Top-N Logik einbauen
                data_to_fit = X[[col]].copy()
                top_n_list = None
                if method == "TOP_N_PLUS_ONE_HOT":
                    top_n_list = X[col].value_counts().head(cfg.get("top_n", 10)).index.tolist()
                    data_to_fit[col] = data_to_fit[col].where(data_to_fit[col].isin(top_n_list), "Other")
                
                enc = ce.OneHotEncoder(cols=[col], use_cat_names=True).fit(data_to_fit)
                fitted_encoders[col] = (enc, top_n_list)
            
            # Transformation für Train UND Test
            enc, top_n_list = fitted_encoders[col]
            data_to_transform = X[[col]].copy()
            if top_n_list:
                data_to_transform[col] = data_to_transform[col].where(data_to_transform[col].isin(top_n_list), "Other")
            encoded_parts.append(enc.transform(data_to_transform))

        elif method == "LEAVE_ONE_OUT_TARGET":
            if is_train:
                enc = ce.LeaveOneOutEncoder(cols=[col]).fit(X[[col]], y)
                fitted_encoders[col] = enc
            
            # Auch hier: Nutzt den fitted_encoder für X_test
            encoded_parts.append(fitted_encoders[col].transform(X[[col]]))
            
    X_encoded = pd.concat(encoded_parts, axis=1)
    return X_encoded, fitted_encoders

# run
X_train_final, encoders = apply_encoding(X_train, y_train_log, feature_config, is_train=True)
X_test_final, _ = apply_encoding(X_test, None, feature_config, is_train=False, fitted_encoders=encoders)




In [47]:
# 1. Scaler 
scaler = StandardScaler()

# 2. Fit & Transform for Training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_final), 
    columns=X_train_final.columns, 
    index=X_train_final.index
)

# 3. ONLY Transform for test data

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_final), 
    columns=X_test_final.columns, 
    index=X_test_final.index
)


In [48]:
#save the datasets as csv

# save features
X_train_scaled.to_csv("X_train_final.csv", index=False)
X_test_scaled.to_csv("X_test_final.csv", index=False)

# save target (y)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# save log target
y_train_log.to_csv("y_train_log.csv", index=False)
y_test_log.to_csv("y_test_log.csv", index=False)